# 03 — Multivariate Distributions
**References:** Casella & Berger (2002) Ch. 4 · DeGroot & Schervish (2012) Ch. 3–5

## Narrative thread
```
Joint distribution -> Marginal -> Conditional -> Independence -> Covariance -> Multivariate Normal
```

## 1. Joint, marginal, and conditional distributions

For random variables $X$ and $Y$ with joint PDF $f_{X,Y}(x,y)$:

**Marginal PDFs:**
$$f_X(x) = \int_{-\infty}^{\infty} f_{X,Y}(x,y)\,dy \qquad f_Y(y) = \int_{-\infty}^{\infty} f_{X,Y}(x,y)\,dx$$

**Conditional PDF** of $Y$ given $X = x$:
$$f_{Y|X}(y \mid x) = \frac{f_{X,Y}(x,y)}{f_X(x)}, \quad f_X(x) > 0$$

**Independence:** $X \perp Y \iff f_{X,Y}(x,y) = f_X(x) f_Y(y)$ for all $(x,y)$.

**Law of total expectation (tower property):**
$$E[Y] = E_X[E[Y \mid X]]$$

**Law of total variance:**
$$\text{Var}(Y) = E[\text{Var}(Y \mid X)] + \text{Var}(E[Y \mid X])$$

> The law of total variance decomposes variability into within-group and between-group components — the foundation of ANOVA.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── Law of total expectation verification ─────────────────────────────────
# Y | X=x ~ N(2*x, 1),  X ~ N(0, 1)
# => E[Y] = E[E[Y|X]] = E[2X] = 0
# => Var(Y) = E[Var(Y|X)] + Var(E[Y|X]) = 1 + Var(2X) = 1 + 4 = 5

n = 200_000
X = np.random.normal(0, 1, n)
Y = np.random.normal(2 * X, 1)   # Y | X ~ N(2X, 1)

print('=== Law of Total Expectation and Variance ===')
print(f'  E[Y]    simulated: {Y.mean():.4f}  theoretical: 0')
print(f'  Var(Y)  simulated: {Y.var():.4f}  theoretical: 5')
print()
print('  Decomposition:')
print(f'  E[Var(Y|X)] = E[1] = 1         (within-group variance)')
print(f'  Var(E[Y|X]) = Var(2X) = 4      (between-group variance)')
print(f'  Total = 1 + 4 = 5  ✓')

## 2. Covariance and correlation

$$\text{Cov}(X,Y) = E[(X-\mu_X)(Y-\mu_Y)] = E[XY] - \mu_X\mu_Y$$

$$\rho(X,Y) = \frac{\text{Cov}(X,Y)}{\sigma_X \sigma_Y}, \quad \rho \in [-1, 1]$$

**Key facts:**
- $X \perp Y \Rightarrow \text{Cov}(X,Y) = 0$ but $\text{Cov}(X,Y) = 0 \not\Rightarrow X \perp Y$
- $\text{Var}(X + Y) = \text{Var}(X) + \text{Var}(Y) + 2\,\text{Cov}(X,Y)$
- For $\mathbf{X} = (X_1,\ldots,X_n)^\top$, the **covariance matrix** is $\Sigma_{ij} = \text{Cov}(X_i, X_j)$

In [ ]:
# ── Correlation ≠ independence: nonlinear example ─────────────────────────
n = 500
X = np.random.uniform(-3, 3, n)
Y_dep   = X**2 + np.random.normal(0, 0.5, n)  # dependent but uncorrelated
Y_indep = np.random.normal(0, 3, n)            # independent

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (xi, yi, title) in zip(axes, [
    (X, Y_dep,   f'Y = X² + ε\nCor={np.corrcoef(X,Y_dep)[0,1]:.3f} ≈ 0, but DEPENDENT'),
    (X, Y_indep, f'Y independent of X\nCor={np.corrcoef(X,Y_indep)[0,1]:.3f} ≈ 0, INDEPENDENT'),
    (X, 0.8*X + np.random.normal(0,1,n),
     f'Y = 0.8X + ε\nCor={np.corrcoef(X, 0.8*X+np.random.normal(0,1,n))[0,1]:.3f} ≠ 0, LINEAR dep.'),
]):
    ax.scatter(xi, yi, s=10, alpha=0.4, color='#4361ee')
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.set_title(title, fontsize=10)

plt.suptitle('Covariance = 0 ≠ Independence (only equivalent for Normal)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. The Multivariate Normal distribution

$\mathbf{X} \sim \mathcal{N}_p(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ with PDF:

$$f(\mathbf{x}) = \frac{1}{(2\pi)^{p/2} |\boldsymbol{\Sigma}|^{1/2}} \exp\!\left(-\frac{1}{2}(\mathbf{x} - \boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1} (\mathbf{x} - \boldsymbol{\mu})\right)$$

**Fundamental properties:**
1. **Marginals are normal:** $X_i \sim \mathcal{N}(\mu_i, \Sigma_{ii})$
2. **Conditionals are normal:** $X_1 \mid X_2 = x_2 \sim \mathcal{N}(\mu_{1|2}, \Sigma_{1|2})$ where
$$\mu_{1|2} = \mu_1 + \Sigma_{12}\Sigma_{22}^{-1}(x_2 - \mu_2), \quad \Sigma_{1|2} = \Sigma_{11} - \Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21}$$
3. **Uncorrelated $\Leftrightarrow$ Independent** (only for MVN!)
4. **Linear combinations:** $\mathbf{A}\mathbf{X} \sim \mathcal{N}(\mathbf{A}\boldsymbol{\mu}, \mathbf{A}\boldsymbol{\Sigma}\mathbf{A}^\top)$
5. **Quadratic forms:** $(\mathbf{X} - \boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1}(\mathbf{X} - \boldsymbol{\mu}) \sim \chi^2(p)$

In [ ]:
# ── Multivariate Normal: contour plots and conditional distributions ──────
mu = np.array([0, 0])
rhos = [0.0, 0.7, -0.9]

x_grid = np.linspace(-4, 4, 200)
y_grid = np.linspace(-4, 4, 200)
XX, YY = np.meshgrid(x_grid, y_grid)
pos = np.dstack((XX, YY))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for col, rho in enumerate(rhos):
    Sigma = np.array([[1, rho], [rho, 1]])
    mv_norm = stats.multivariate_normal(mu, Sigma)
    Z = mv_norm.pdf(pos)

    # Contour plot
    ax = axes[0, col]
    ct = ax.contourf(XX, YY, Z, levels=15, cmap='Blues')
    ax.contour(XX, YY, Z, levels=15, colors='white', linewidths=0.5, alpha=0.5)
    ax.set_title(f'$\\mathcal{{N}}_2(\\mathbf{{0}}, \\Sigma)$, $\\rho={rho}$\n'
                 f'Ellipse axes aligned to eigenvectors of $\\Sigma$')
    ax.set_xlabel('$X_1$'); ax.set_ylabel('$X_2$')

    # Conditional distribution: X1 | X2 = x2
    ax2 = axes[1, col]
    x1_vals = np.linspace(-4, 4, 300)
    x2_conditions = [-1.5, 0, 1.5]
    colors_cond = ['#4361ee', '#06d6a0', '#f72585']
    for x2_given, c in zip(x2_conditions, colors_cond):
        mu_cond   = mu[0] + rho * (x2_given - mu[1])
        var_cond  = 1 - rho**2
        pdf_cond  = stats.norm.pdf(x1_vals, mu_cond, np.sqrt(var_cond))
        ax2.plot(x1_vals, pdf_cond, lw=2, color=c,
                 label=f'$X_2={x2_given}$: $\\mu_{{1|2}}={mu_cond:.1f}$')
    ax2.set_title(f'Conditional $X_1 \\mid X_2=x_2$, $\\rho={rho}$\n'
                  f'$\\sigma^2_{{1|2}} = 1-\\rho^2 = {1-rho**2:.2f}$')
    ax2.set_xlabel('$x_1$'); ax2.set_ylabel('$f(x_1 \\mid x_2)$')
    ax2.legend(fontsize=8)

plt.suptitle('Multivariate Normal — Contours and Conditional Distributions', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Order statistics

Given i.i.d. $X_1,\ldots,X_n$ with PDF $f$ and CDF $F$, the **order statistics** are $X_{(1)} \leq \cdots \leq X_{(n)}$.

The PDF of the $k$-th order statistic:
$$f_{X_{(k)}}(x) = \frac{n!}{(k-1)!(n-k)!} [F(x)]^{k-1}[1-F(x)]^{n-k} f(x)$$

**Special cases:**
- $X_{(1)} = \min$: $f_{(1)}(x) = n[1-F(x)]^{n-1}f(x)$
- $X_{(n)} = \max$: $f_{(n)}(x) = n[F(x)]^{n-1}f(x)$

Order statistics are central to **nonparametric inference** (notebook 12) and to the derivation of complete sufficient statistics (notebook 05).

| Takeaway | Key idea |
|---|---|
| Marginal integrates out the other variable | $f_X(x) = \int f_{X,Y}(x,y)\,dy$ |
| MVN: uncorrelated $\Leftrightarrow$ independent | True only for Gaussians |
| Conditional MVN is explicit | Schur complement gives conditional variance |
| Order statistics are essential | Nonparametrics, sufficient statistics |

**Next:** notebook 04 — convergence theorems (LLN, CLT) which justify large-sample inference.